# What's in an Avocado Toast: A Supply Chain Analysis

![](avocado_wallpaper.jpeg)

You find yourself in London, crafting a delectable avocado toast, a dish that has risen dramatically in popularity on breakfast menus since the 2010s. This straightforward recipe requires just five ingredients: a ripe avocado, half a lemon, a generous pinch of salt flakes, two slices of sourdough bread, and a good drizzle of extra virgin olive oil. Most of these ingredients are now staples in grocery stores, and as you will find with this project, that is no small feat!

In this project, you'll conduct a supply chain analysis of three ingredients used in avocado toast using the Open Food Facts database. This database contains extensive, openly-sourced information on various foods, including their origins. Through this analysis, you will gain an in-depth understanding of the complex supply chain involved in producing a single dish.

Three pairs of files are provided in the data folder:
- A CSV file for each ingredient, such as `avocado.csv`, with data about each food item and countries of origin.
- A TXT file for each ingredient, such as `relevant_avocado_categories`, containing only the category tags of interest for that food.

Here are some other key points about these files:
- Some of the rows of data in each of the three CSV files do not contain relevant data for your investigation. In each dataset, you will need to filter out rows with irrelevant data, based on values in the `categories_tags` column. Examples of categories are fruits, vegetables, and fruit-based oils. Filter the DataFrame to include only rows where `categories_tags` contains one of the tags in the relevant categories for that ingredient.
- Each row of data usually has multiple category tags in the `categories_tags` column.
There is a column in each CSV file called `origins_tags`, which contains strings for the country of origin of each item.

After completing this project, you'll be armed with a list of ingredients and their countries of origin and be well-positioned to launch into other analyses that explore how long, on average, these ingredients spend at sea.

[Open Food Facts database](https://world.openfoodfacts.org/)

In [1]:
# Libraries imports
import pandas as pd
import numpy as np 

# Reading datasets into machine
avoc = pd.read_csv("data/avocado.csv", sep="\t")

# Reading the text file for relevant avocado categories
with open("data/relevant_avocado_categories.txt") as file: 
    relevant_avocado_categories = file.read().splitlines()
    file.close()


print(avoc.info())
print("the shape of the avocado data is: ", avoc.shape)
print(avoc.columns.unique()[:30]) # To check and confirm the colnames provided to be consistent with the colnames in the dataset.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1785 entries, 0 to 1784
Columns: 184 entries, code to data_sources
dtypes: float64(58), int64(1), object(125)
memory usage: 2.5+ MB
None
the shape of the avocado data is:  (1785, 184)
Index(['code', 'lc', 'product_name_de', 'product_name_el', 'product_name_en',
       'product_name_es', 'product_name_fi', 'product_name_fr',
       'product_name_id', 'product_name_it', 'product_name_lt',
       'product_name_lv', 'product_name_nb', 'product_name_nl',
       'product_name_pl', 'product_name_ro', 'product_name_sl',
       'product_name_sv', 'generic_name_de', 'generic_name_en',
       'generic_name_es', 'generic_name_fr', 'generic_name_sv', 'quantity',
       'serving_size', 'packaging', 'packaging_tags', 'brands', 'brands_tags',
       'brand_owner'],
      dtype='object')


In [2]:
# Filtering the avocado data by relevant categories

relevant_columns = ['code', 'lc', 'product_name_en', 'quantity', 'serving_size', 'packaging_tags', 'brands', 'brands_tags', 'categories_tags', 'labels_tags', 'countries', 'countries_tags', 'origins', 'origins_tags']

avocado = avoc[relevant_columns]

avocado.head()



,code,lc,product_name_en,quantity,serving_size,packaging_tags,brands,brands_tags,categories_tags,labels_tags,countries,countries_tags,origins,origins_tags
0,0059749979702,fr,NaN,NaN,NaN,NaN,Naturalia,naturalia,"en:plant-based-foods-and-beverages,en:plant-ba...",NaN,Canada,en:canada,NaN,NaN
1,7610095131409,en,NaN,NaN,NaN,NaN,Zweifel,zweifel,"en:snacks,en:salty-snacks,en:appetizers,en:chi...","en:vegetarian,en:vegan","Switzerland, World","en:switzerland,en:world",NaN,NaN
2,4005514005578,en,Gelbe Linse Avocado Brotaufstrich,NaN,NaN,NaN,Tartex,tartex,de:abendbrotsufstrich,"en:organic,en:eu-organic,en:eg-oko-verordnung",Germany,en:germany,NaN,NaN
3,0879890002513,en,Avocado toast chili lime,NaN,NaN,NaN,NaN,NaN,NaN,NaN,United States,en:united-states,NaN,NaN
4,0223086613685,en,Avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,United States,en:united-states,NaN,NaN


In [3]:
# inspecting the colnames and their info for effective cleaning, while checking the info in the categories_tags 

avocado.info()
avocado["categories_tags"].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1785 entries, 0 to 1784
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   code             1785 non-null   object
 1   lc               1785 non-null   object
 2   product_name_en  1273 non-null   object
 3   quantity         276 non-null    object
 4   serving_size     473 non-null    object
 5   packaging_tags   146 non-null    object
 6   brands           674 non-null    object
 7   brands_tags      674 non-null    object
 8   categories_tags  752 non-null    object
 9   labels_tags      306 non-null    object
 10  countries        1779 non-null   object
 11  countries_tags   1779 non-null   object
 12  origins          58 non-null     object
 13  origins_tags     58 non-null     object
dtypes: object(14)
memory usage: 195.4+ KB


0    en:plant-based-foods-and-beverages,en:plant-ba...
1    en:snacks,en:salty-snacks,en:appetizers,en:chi...
2                                de:abendbrotsufstrich
3                                                  NaN
4                                                  NaN
Name: categories_tags, dtype: object

In [4]:
# spliting the categories_tags column for effective EDA
avocado["categories_tags"] = avocado["categories_tags"].str.split(",")
print(avocado.head())


            code  lc  ... origins origins_tags
0  0059749979702  fr  ...     NaN          NaN
1  7610095131409  en  ...     NaN          NaN
2  4005514005578  en  ...     NaN          NaN
3  0879890002513  en  ...     NaN          NaN
4  0223086613685  en  ...     NaN          NaN

[5 rows x 14 columns]


In [5]:
# dropping rows with null values in the categories_tags column
avocado = avocado.dropna(subset=["categories_tags"])
avocado.head()

,code,lc,product_name_en,quantity,serving_size,packaging_tags,brands,brands_tags,categories_tags,labels_tags,countries,countries_tags,origins,origins_tags
0,0059749979702,fr,NaN,NaN,NaN,NaN,Naturalia,naturalia,"[en:plant-based-foods-and-beverages, en:plant-...",NaN,Canada,en:canada,NaN,NaN
1,7610095131409,en,NaN,NaN,NaN,NaN,Zweifel,zweifel,"[en:snacks, en:salty-snacks, en:appetizers, en...","en:vegetarian,en:vegan","Switzerland, World","en:switzerland,en:world",NaN,NaN
2,4005514005578,en,Gelbe Linse Avocado Brotaufstrich,NaN,NaN,NaN,Tartex,tartex,[de:abendbrotsufstrich],"en:organic,en:eu-organic,en:eg-oko-verordnung",Germany,en:germany,NaN,NaN
5,3662994002063,fr,NaN,3 fruits,NaN,NaN,la compagnie des fruits mûrs,la-compagnie-des-fruits-murs,"[en:plant-based-foods-and-beverages, en:plant-...",NaN,France,en:france,NaN,NaN
6,8437013031011,fr,NaN,1 kg,NaN,NaN,NaN,NaN,"[en:plant-based-foods-and-beverages, en:plant-...",NaN,France,en:france,NaN,NaN


In [6]:
# inspecting the items in relevant_avocado_categories
print(relevant_avocado_categories)

['en:avocadoes', 'en:avocados', 'en:fresh-foods', 'en:fresh-vegetables', 'en:fruchte', 'en:fruits', 'en:raw-green-avocados', 'en:tropical-fruits', 'en:tropische-fruchte', 'en:vegetables-based-foods', 'fr:hass-avocados']


In [7]:
# filtering the avocado based on the relevant_avocado_categories 
# To return on those rows from avocado where categories_tags contains any of the values in the reference list provided as relevant_avocado_categories 

avocado = avocado[avocado["categories_tags"].apply(lambda x: 
                                 any([i for i in x if i in relevant_avocado_categories]) )]
avocado.head()

,code,lc,product_name_en,quantity,serving_size,packaging_tags,brands,brands_tags,categories_tags,labels_tags,countries,countries_tags,origins,origins_tags
5,3662994002063,fr,NaN,3 fruits,NaN,NaN,la compagnie des fruits mûrs,la-compagnie-des-fruits-murs,"[en:plant-based-foods-and-beverages, en:plant-...",NaN,France,en:france,NaN,NaN
6,8437013031011,fr,NaN,1 kg,NaN,NaN,NaN,NaN,"[en:plant-based-foods-and-beverages, en:plant-...",NaN,France,en:france,NaN,NaN
14,4016249238155,de,NaN,135g,100g,de:gläschen,Allos,allos,"[en:plant-based-foods-and-beverages, en:plant-...","en:organic,en:vegetarian,en:eu-organic,en:no-g...",Deutschland,en:germany,Europäische Union,en:european-union
17,8718963381532,de,NaN,NaN,NaN,NaN,NaN,NaN,"[en:plant-based-foods-and-beverages, en:plant-...",NaN,Deutschland,en:germany,NaN,NaN
23,8436002746707,es,NaN,NaN,NaN,NaN,NaN,NaN,"[en:plant-based-foods-and-beverages, en:plant-...",NaN,España,en:spain,NaN,NaN


In [8]:
# inspect the dataframe in the countries column to enable us know how to filter for the United_Kingdom 

avocado["countries"].unique() 

array(['France', 'Deutschland', 'España', 'Belgique', 'Germany',
       'Germany, Italy', 'France, Allemagne, Suisse', 'Canada',
       'France, Netherlands', 'Frankrijk, Nederland', 'France, Germany',
       'Australia', 'United States', 'United Kingdom', 'Nederland',
       'France, Pays-Bas', 'Canada, United States', 'Canada, France',
       'Deutschland, Schweiz', 'Norway',
       'Germany, Switzerland, United Kingdom, United States',
       'Ireland, United Kingdom', 'Belgique, France', 'Ireland', 'Norge',
       'Latvija', 'Royaume-Uni', 'Ukraine', 'France, Royaume-Uni',
       'China, México, Estados Unidos', 'Belgien, Deutschland', 'Finland',
       'en:Deutschland', 'Italia', 'Italy', 'Suisse', 'Lietuva'],
      dtype=object)

In [9]:
# filtering the avocado to return rows for United Kingdom only

avocado_uk = avocado[avocado["countries"] == "United Kingdom"]
print("the current dimension of avocado_uk is:", avocado_uk.shape, "compare to avocado:", avocado.shape )

the current dimension of avocado_uk is: (13, 14) compare to avocado: (182, 14)


In [10]:
avocado_uk.head()

,code,lc,product_name_en,quantity,serving_size,packaging_tags,brands,brands_tags,categories_tags,labels_tags,countries,countries_tags,origins,origins_tags
361,00985833,en,Avacado,650 g,NaN,NaN,Marks & Spencer,marks-spencer,"[en:plant-based-foods-and-beverages, en:plant-...",NaN,United Kingdom,en:united-kingdom,Peru,en:peru
381,00040464,en,Avocado,NaN,NaN,NaN,NaN,NaN,"[en:plant-based-foods-and-beverages, en:plant-...",NaN,United Kingdom,en:united-kingdom,NaN,NaN
414,4088600100173,en,Avocado,100 g,NaN,en:mixed-plastic-unknown,Aldi,aldi,"[en:plant-based-foods-and-beverages, en:plant-...",NaN,United Kingdom,en:united-kingdom,NaN,NaN
468,01307351,en,Avacados organic,NaN,NaN,"en:card-tray,en:ldpe-bag",Sainsbury’s SO organic,sainsbury-s-so-organic,"[en:plant-based-foods-and-beverages, en:plant-...","en:organic,en:eu-organic,en:non-eu-agriculture...",United Kingdom,en:united-kingdom,NaN,NaN
508,5057172125395,en,Just Essentials Avocados,4pack,NaN,en:mixed-plastic-film-packet-to-recycle,Asda,asda,"[en:plant-based-foods-and-beverages, en:plant-...","en:class-i,en:contains-stones",United Kingdom,en:united-kingdom,Peru,en:peru


In [11]:
# avocado_uk.groupby("origins_tags")["origins_tags"].value_counts()

top_avocado_origin = avocado_uk["origins_tags"].value_counts().index[0]
top_avocado_origin = top_avocado_origin.lstrip("en:").replace("-"," ")
# top_avocado_origin = top_avocado_origin.replace("-", " ")
print(top_avocado_origin)

peru


In [13]:
# creating a general function to perform similar task (DRY)

# Reading datasets into the machine
avoc = pd.read_csv("data/avocado.csv", sep="\t")

# Reading the text file for relevant avocado categories
with open("data/relevant_avocado_categories.txt") as file: 
    relevant_avocado_categories = file.read().splitlines()
        
def read_and_filter_data(filename, relevant_categories): 
 

# Filtering the avocado data by relevant categories
    relevant_columns = ['code', 'lc', 'product_name_en', 'quantity', 'serving_size', 'packaging_tags', 'brands', 'brands_tags', 'categories_tags', 'labels_tags', 'countries', 'countries_tags', 'origins', 'origins_tags']

    avocado = avoc[relevant_columns]

# spliting the categories_tags column for effective manipulations
    avocado["categories_tags"] = avocado["categories_tags"].str.split(",")

# dropping rows with null values in the categories_tags column
    avocado = avocado.dropna(subset=["categories_tags"])

# filtering the avocado based on the relevant_avocado_categories 
# To return on those rows from avocado where categories_tags contains any of the values in the reference list provided as relevant_avocado_categories 

    avocado = avocado[avocado["categories_tags"].apply(lambda x: 
                                 any([i for i in x if i in relevant_avocado_categories]) )]


# filtering the avocado to return rows for United Kingdom only
    avocado_uk = avocado[avocado["countries"] == "United Kingdom"]


# getting the desired outcome of the country with the top avocado origin in UK
# stripping the characters before the country name or hyphens

    top_avocado_origin = avocado_uk["origins_tags"].value_counts().index[0]
    top_avocado_origin = top_avocado_origin.lstrip("en:").replace("-"," ")
    
    return top_avocado_origin
top_avocado_origin =read_and_filter_data(
    avoc,
    relevant_avocado_categories
)
print(top_avocado_origin)

peru


In [14]:
# top olive oil origin
# creating a general function to perform similar task (DRY)

    # Reading datasets into the machine
olive_oil = pd.read_csv("data/olive_oil.csv", sep="\t")

# Reading the text file for relevant avocado categories
with open("data/relevant_olive_oil_categories.txt") as file: 
    relevant_olive_oil_categories = file.read().splitlines()

def read_and_filter_data(df, relevant_categories): 
    # Filtering the sourdough data by relevant categories
    relevant_columns = [
        'code', 'lc', 'product_name_en', 'quantity', 'serving_size', 'packaging_tags', 
        'brands', 'brands_tags', 'categories_tags', 'labels_tags', 'countries', 
        'countries_tags', 'origins', 'origins_tags'
    ]
    df = df[relevant_columns]

    # splitting the categories_tags column for effective EDA
    df["categories_tags"] = df["categories_tags"].str.split(",")

    # dropping rows with null values in the categories_tags column
    df = df.dropna(subset=["categories_tags"])

    # filtering the sourdough based on the relevant_sourdough_categories 
    df = df[df["categories_tags"].apply(lambda x: 
                                 any(i in relevant_categories for i in x))]

    # filtering the sourdough to return rows for United Kingdom only
    df_uk = df[df["countries"] == "United Kingdom"]

    # getting the desired outcome of the country with the top sourdough origin in UK
    # stripping the characters before the country name or hyphens
    top_olive_oil_origin = df_uk["origins_tags"].value_counts().index[0]
    top_olive_oil_origin = top_olive_oil_origin.lstrip("en:").replace("-"," ")
    
    return top_olive_oil_origin

top_olive_oil_origin = read_and_filter_data(
    olive_oil, 
    relevant_olive_oil_categories
)
print(top_olive_oil_origin)

greece


In [15]:
import pandas as pd

# Reading datasets into the machine
sourdough = pd.read_csv("data/sourdough.csv", sep="\t")

# Reading the text file for relevant sourdough categories
with open("data/relevant_sourdough_categories.txt") as file: 
    relevant_sourdough_categories = file.read().splitlines()

def read_and_filter_data(df, relevant_categories): 
    # Filtering the sourdough data by relevant categories
    relevant_columns = [
        'code', 'lc', 'product_name_en', 'quantity', 'serving_size', 'packaging_tags', 
        'brands', 'brands_tags', 'categories_tags', 'labels_tags', 'countries', 
        'countries_tags', 'origins', 'origins_tags'
    ]
    df = df[relevant_columns]

    # splitting the categories_tags column for effective EDA
    df["categories_tags"] = df["categories_tags"].str.split(",")

    # dropping rows with null values in the categories_tags column
    df = df.dropna(subset=["categories_tags"])

    # filtering the sourdough based on the relevant_sourdough_categories 
    df = df[df["categories_tags"].apply(lambda x: 
                                 any(i in relevant_categories for i in x))]

    # filtering the sourdough to return rows for United Kingdom only
    df_uk = df[df["countries"] == "United Kingdom"]

    # getting the desired outcome of the country with the top sourdough origin in UK
    # stripping the characters before the country name or hyphens
    top_sourdough_origin = df_uk["origins_tags"].value_counts().index[0]
    top_sourdough_origin = top_sourdough_origin.lstrip("en:").replace("-"," ")
    
    return top_sourdough_origin

top_sourdough_origin = read_and_filter_data(
 sourdough,
 relevant_sourdough_categories
)
print(top_sourdough_origin)

united kingdom
